# Cyclic similarity across recurrent depth

How do residual streams and attention patterns repeat across virtual layers?
Each of three pairwise measurements has **main**, **appendix**, and **all** figures
using the shared model groups in `experiments/paper_models.py`. Main and appendix
have four square heatmaps in one row; all stacks these rows. Parcae 770M appears
in the main group, with Parcae 140M and 370M in the appendix.

Every model uses **eight recurrences**. `mode="quick"` defaults to one seeded
example and saves figures only in the run directory. `mode="full"` defaults to
256 examples and also updates the paper figures. SLURM selects full mode; an
explicit `limit` overrides the sample count. Both modes use the same pairs and plots.

In [ ]:
mode = "quick"  # "quick" saves locally; "full" also updates paper figures.
dataset = "gsm8k"  # "gsm8k" or "hellaswag"
recurrences = 8  # Applied to every model in the shared paper groups.
model_recurrences = None  # Optional explicit [(model, recurrences), ...] subset.
limit = None  # Default: 1 (quick), 256 (full); 0 means the entire split.
seed = 42
device = "auto"
dtype = "auto"
output_dir = None

cache_dir = None  # Defaults to LOOPED_INTERNALS_CACHE_DIR or /data-gauss/hugbla/looped-internals-cache.
cache_mode = None  # Environment default; "reuse", "require" (no inference), or "off".
analysis_version = 1  # Bump when this notebook's scientific reduction changes.
source_run = None  # Optional completed run directory with matching scientific settings.

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
from dataclasses import asdict
from importlib.metadata import version

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.ticker import MaxNLocator

from experiments.cache import ExperimentCache
from experiments.paper_models import paper_model_pairs
from experiments.saved_runs import SavedRun
from experiments.support import (
    COLORMAPS,
    MODEL_LABELS,
    collect_metrics,
    configure_experiment,
    export_paper_figures,
    model_subplots,
    save_figure,
    save_results,
    shorten_label,
)

plt.style.use(ROOT / "plots" / "matlab.mplstyle")

# Resolve shared defaults after Papermill overrides; collect repeated panels once.
model_recurrences, figure_pairs = paper_model_pairs(recurrences, model_recurrences)
sample_limit = limit if limit is not None else (1 if mode == "quick" else 256)
configs = {
    (name, recurrences): configure_experiment(
        experiment="cyclic_similarity",
        dataset=dataset,
        mode=mode,
        models=[name],
        num_steps=recurrences,
        limit=sample_limit,
        seed=seed,
        device=device,
        dtype=dtype,
        output_dir=output_dir,
    )
    for name, recurrences in model_recurrences
}
config = next(iter(configs.values()))

result_cache = ExperimentCache(cache_dir, cache_mode)

In [ ]:
PAIRWISE_METRICS = {
    "pairwise_residual_cosine_sim": "Residual cosine similarity",
    "pairwise_residual_diff_norm": "Residual Euclidean difference norm",
    "pairwise_attn_pattern_diff_norm": "Attention-pattern Frobenius difference norm",
}


def summarize_similarity(run):
    """Average tokens or heads within one example, keeping both virtual-layer axes."""
    return {
        field: getattr(run, field).float().mean(-1).tolist()
        for field in PAIRWISE_METRICS
    }

## Generate and reduce

Each required pair runs on the same seeded examples. Collection releases raw
activations after each forward pass; once a pair finishes, average its per-example
matrices and release its model and collected metrics. Residual spectral entropy
and successive/final-state comparisons are disabled because these figures only
use pairwise measurements.

Save the mean matrices and run metadata in `overview.pt`, with one `.npy` file per
pair and metric. Figure cells reuse these means without inference or further reduction.

In [ ]:
settings = asdict(config)
del settings["models"], settings["num_steps"]
settings.update(
    model_recurrences=model_recurrences,
    figure_pairs=figure_pairs,
    output_dir=str(config.output_dir),
)
settings["versions"] = {
    name: version(name) for name in ("torch", "transformer-lens", "transformers")
}
settings["cache"] = result_cache.settings
previous = SavedRun(source_run, settings, analysis_version=analysis_version)
previous.check_pairs(settings)
settings.update(analysis_version=analysis_version, source_run=source_run)
run_dir = save_results({}, settings, config.output_dir)

summaries, run_info = {}, {}
for pair, pair_config in configs.items():
    if pair in previous:
        summaries[pair] = previous.data["summaries"][pair]
        run_info[pair] = previous.info(pair)
        for metric, matrix in summaries[pair].items():
            np.save(run_dir / f"{pair[0]}-{pair[1]}-steps-{metric}.npy", matrix)
        continue
    name, recurrences = pair
    results, info = collect_metrics(
        pair_config,
        summarize_similarity,
        cache=result_cache,
        analysis_version=analysis_version,
    )
    summaries[pair] = {
        metric: np.mean([example[metric] for example in results[name]], axis=0)
        for metric in PAIRWISE_METRICS
    }
    run_info[pair] = info
    for metric, matrix in summaries[pair].items():
        np.save(run_dir / f"{name}-{recurrences}-steps-{metric}.npy", matrix)
    del results

torch.save(
    {"summaries": summaries, "run": run_info, "figure_pairs": figure_pairs},
    run_dir / "overview.pt",
)
result_cache.write_manifest(run_dir)

## What the heatmaps measure

Both axes follow virtual-layer execution order: recurrence first, then physical
layer, including any prelude and coda. Average tokens (or heads) **within each
example first**, then average examples with equal weight. There are no uncertainty
bands on these mean matrices.

| Measurement | Interpretation | Colour scale |
| --- | --- | --- |
| Residual cosine similarity | Larger values mean better directional alignment | −1 to 1 |
| Residual difference norm | Euclidean distance includes changes in magnitude and direction | Zero to that model's maximum |
| Attention difference norm | Frobenius distance between attention patterns | Zero to that model's maximum |

Each heatmap has its own colourbar. Distance scales vary by model and are reused
when the same model appears in another figure. Axis and colourbar labels appear
only at the row edges; tick values remain visible on every panel. Model titles
omit the recurrence count, which is common to the figures and belongs in the caption.

In [ ]:
def plot_similarity(metric, *, group):
    """Plot square heatmaps with separate model scales in the shared paper layout."""
    pairs = figure_pairs[group]
    if not pairs:
        return
    with plt.style.context(ROOT / "plots" / "matlab.mplstyle"):
        fig, axes = model_subplots(len(pairs), height_per_subplot=1.75)
        cmap = COLORMAPS[
            "attention_patterns" if "attn" in metric else "residual_streams"
        ]
        for index, (ax, pair) in enumerate(zip(axes, pairs, strict=True)):
            name, _ = pair
            matrix = summaries[pair][metric]
            cosine = metric.endswith("cosine_sim")
            artist = ax.imshow(
                matrix,
                cmap=cmap,
                vmin=-1 if cosine else 0,
                vmax=1 if cosine else float(matrix.max()),
            )
            ax.set(title=MODEL_LABELS[name], xlabel="Virtual layer")
            if ax.get_subplotspec().is_first_col():
                ax.set_ylabel("Virtual layer")
            ax.xaxis.set_major_locator(MaxNLocator(nbins=3, integer=True))
            ax.yaxis.set_major_locator(MaxNLocator(nbins=3, integer=True))
            ax.grid(False)
            colorbar = fig.colorbar(artist, ax=ax, fraction=0.045, pad=0.035, aspect=25)
            colorbar.locator = MaxNLocator(nbins=3)
            colorbar.update_ticks()
            if ax.get_subplotspec().is_last_col() or index == len(pairs) - 1:
                colorbar.set_label(shorten_label(PAIRWISE_METRICS[metric]))
        save_figure(fig, run_dir, f"{metric}_{group}", bbox_inches=None)

## Residual cosine similarity

In [ ]:
plot_similarity("pairwise_residual_cosine_sim", group="main")

In [ ]:
plot_similarity("pairwise_residual_cosine_sim", group="appendix")

In [ ]:
plot_similarity("pairwise_residual_cosine_sim", group="all")

## Residual difference norm

In [ ]:
plot_similarity("pairwise_residual_diff_norm", group="main")

In [ ]:
plot_similarity("pairwise_residual_diff_norm", group="appendix")

In [ ]:
plot_similarity("pairwise_residual_diff_norm", group="all")

## Attention difference norm

In [ ]:
plot_similarity("pairwise_attn_pattern_diff_norm", group="main")

In [ ]:
plot_similarity("pairwise_attn_pattern_diff_norm", group="appendix")

In [ ]:
plot_similarity("pairwise_attn_pattern_diff_norm", group="all")

In [ ]:
print(run_dir)
paper_dir = export_paper_figures(config, run_dir, dataset=config.dataset)
if paper_dir is not None:
    print(f"Saved paper figures to {paper_dir}")